In [31]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5ForConditionalGeneration, T5Tokenizer
from sklearn.model_selection import train_test_split
from rouge_score import rouge_scorer
import numpy as np

### Load Data

In [18]:
df = pd.read_csv("prompt_answer_pairs_clean.csv")[["prompt", "answer"]].dropna()

# Convert to strings
df["prompt"] = df["prompt"].astype(str).str.strip()
df["answer"] = df["answer"].astype(str).str.strip()

# 70/15/15
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

### Tokenize data

In [19]:
def tokenize_data(df):
  # Input is the answer, tokenizes answer text
  inputs = tokenizer(
      ["predict prompt: " + text for text in df["answer"].tolist()],
      max_length=512,
      padding="max_length",
      truncation=True,
      return_tensors="pt"
    )
  # Target is the prompt, tokenizes prompt text
  targets = tokenizer(
      df["prompt"].tolist(),
      max_length=512,
      padding="max_length",
      truncation=True,
      return_tensors="pt"
  )

  labels = targets["input_ids"]
  # Replace padding tokens with -100 so model does not try to predict them
  labels[labels == tokenizer.pad_token_id] = -100

  # Create tokenized dataset
  dataset = torch.utils.data.TensorDataset(
      inputs["input_ids"],
      inputs["attention_mask"],
      labels
  )
  return dataset

In [20]:
# Load tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-small")
# Model is a T5 language model
# T5 is a seq2seq model that is pretrained using a  using a denoising objective
model     = T5ForConditionalGeneration.from_pretrained("t5-small").to(torch.device("cuda"))

# Tokenize all datasets
train_dataset = tokenize_data(train_df)
val_dataset   = tokenize_data(val_df)
test_dataset  = tokenize_data(test_df)

# DataLoader does batching, shuffling and makes dataset iterable
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=8)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

### Evaluate how model is doing on validation set

In [21]:
def evaluate(loader):
  # Switch model to evaluation mode
  model.eval()
  total_loss = 0
  # Do not track gradients to save memory
  with torch.no_grad():
    # For every batch accumulate the loss
    for batch in loader:
        input_ids, attention_mask, labels = batch
        input_ids      = input_ids.to(torch.device("cuda"))
        attention_mask = attention_mask.to(torch.device("cuda"))
        labels         = labels.to(torch.device("cuda"))
        loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss
        total_loss += loss.item()
  # Return the average loss per epoch
  return total_loss / len(loader)

### Train Model

In [22]:
# Load optimizer
optimizer   = torch.optim.AdamW(model.parameters(), lr=5e-5)
best_val    = float("inf")

for epoch in range(5):
  # Switch model to train mode
  model.train()
  total_train_loss = 0

  for i, batch in enumerate(train_loader):
    input_ids, attention_mask, labels = batch
    input_ids      = input_ids.to(torch.device("cuda"))
    attention_mask = attention_mask.to(torch.device("cuda"))
    labels         = labels.to(torch.device("cuda"))

    # Clear gradient from previous batch
    optimizer.zero_grad()
    # Calculates loss using T5 model
    loss = model(input_ids=input_ids,
                  attention_mask=attention_mask,
                  labels=labels).loss
    # Backpropagation
    loss.backward()
    # Gradient clipping (caps gradients at 1)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    # Updates model weights using gradients calculated by loss.backward
    optimizer.step()

    total_train_loss += loss.item()

    if (i + 1) % 50 == 0:
      print(f"  Epoch {epoch+1} | Step {i+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

  avg_train = total_train_loss / len(train_loader)
  avg_val   = evaluate(val_loader)
  print(f"\nEpoch {epoch+1}/5 | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")

  # Gets best model by updating model saved when average validation loss is less
  if avg_val < best_val:
    best_val = avg_val
    model.save_pretrained("prompt_predictor")
    tokenizer.save_pretrained("prompt_predictor")

  Epoch 1 | Step 50/705 | Loss: 4.7574
  Epoch 1 | Step 100/705 | Loss: 3.3379
  Epoch 1 | Step 150/705 | Loss: 3.3254
  Epoch 1 | Step 200/705 | Loss: 1.3641
  Epoch 1 | Step 250/705 | Loss: 4.7970
  Epoch 1 | Step 300/705 | Loss: 2.2657
  Epoch 1 | Step 350/705 | Loss: 3.8405
  Epoch 1 | Step 400/705 | Loss: 2.7726
  Epoch 1 | Step 450/705 | Loss: 3.3714
  Epoch 1 | Step 500/705 | Loss: 3.2817
  Epoch 1 | Step 550/705 | Loss: 2.8838
  Epoch 1 | Step 600/705 | Loss: 1.3469
  Epoch 1 | Step 650/705 | Loss: 3.1491
  Epoch 1 | Step 700/705 | Loss: 3.8704

Epoch 1/5 | Train Loss: 3.4801 | Val Loss: 2.8563


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Epoch 2 | Step 50/705 | Loss: 2.7150
  Epoch 2 | Step 100/705 | Loss: 2.7047
  Epoch 2 | Step 150/705 | Loss: 2.0709
  Epoch 2 | Step 200/705 | Loss: 2.6637
  Epoch 2 | Step 250/705 | Loss: 3.2059
  Epoch 2 | Step 300/705 | Loss: 3.7266
  Epoch 2 | Step 350/705 | Loss: 3.1806
  Epoch 2 | Step 400/705 | Loss: 3.4544
  Epoch 2 | Step 450/705 | Loss: 3.2533
  Epoch 2 | Step 500/705 | Loss: 4.3527
  Epoch 2 | Step 550/705 | Loss: 1.5504
  Epoch 2 | Step 600/705 | Loss: 2.3846
  Epoch 2 | Step 650/705 | Loss: 2.9682
  Epoch 2 | Step 700/705 | Loss: 3.8205

Epoch 2/5 | Train Loss: 2.9648 | Val Loss: 2.7003


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Epoch 3 | Step 50/705 | Loss: 3.7745
  Epoch 3 | Step 100/705 | Loss: 2.8314
  Epoch 3 | Step 150/705 | Loss: 3.4076
  Epoch 3 | Step 200/705 | Loss: 3.7038
  Epoch 3 | Step 250/705 | Loss: 3.0107
  Epoch 3 | Step 300/705 | Loss: 2.6650
  Epoch 3 | Step 350/705 | Loss: 2.4009
  Epoch 3 | Step 400/705 | Loss: 3.2883
  Epoch 3 | Step 450/705 | Loss: 2.9274
  Epoch 3 | Step 500/705 | Loss: 3.8051
  Epoch 3 | Step 550/705 | Loss: 1.8215
  Epoch 3 | Step 600/705 | Loss: 2.2790
  Epoch 3 | Step 650/705 | Loss: 3.0498
  Epoch 3 | Step 700/705 | Loss: 3.6349

Epoch 3/5 | Train Loss: 2.8131 | Val Loss: 2.6110


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Epoch 4 | Step 50/705 | Loss: 3.8249
  Epoch 4 | Step 100/705 | Loss: 1.9826
  Epoch 4 | Step 150/705 | Loss: 2.3684
  Epoch 4 | Step 200/705 | Loss: 2.6711
  Epoch 4 | Step 250/705 | Loss: 3.4329
  Epoch 4 | Step 300/705 | Loss: 2.1896
  Epoch 4 | Step 350/705 | Loss: 1.4883
  Epoch 4 | Step 400/705 | Loss: 3.9075
  Epoch 4 | Step 450/705 | Loss: 3.1292
  Epoch 4 | Step 500/705 | Loss: 2.7497
  Epoch 4 | Step 550/705 | Loss: 0.6969
  Epoch 4 | Step 600/705 | Loss: 1.0720
  Epoch 4 | Step 650/705 | Loss: 1.7054
  Epoch 4 | Step 700/705 | Loss: 1.6032

Epoch 4/5 | Train Loss: 2.7130 | Val Loss: 2.5484


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Epoch 5 | Step 50/705 | Loss: 3.5781
  Epoch 5 | Step 100/705 | Loss: 3.4010
  Epoch 5 | Step 150/705 | Loss: 0.7100
  Epoch 5 | Step 200/705 | Loss: 4.2381
  Epoch 5 | Step 250/705 | Loss: 2.4952
  Epoch 5 | Step 300/705 | Loss: 0.7745
  Epoch 5 | Step 350/705 | Loss: 3.3909
  Epoch 5 | Step 400/705 | Loss: 2.2122
  Epoch 5 | Step 450/705 | Loss: 2.1986
  Epoch 5 | Step 500/705 | Loss: 2.3246
  Epoch 5 | Step 550/705 | Loss: 3.7623
  Epoch 5 | Step 600/705 | Loss: 1.6072
  Epoch 5 | Step 650/705 | Loss: 3.8695
  Epoch 5 | Step 700/705 | Loss: 3.0163

Epoch 5/5 | Train Loss: 2.6449 | Val Loss: 2.4920


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### Evaulate Model Performace with ROUGE
ROUGE is a metric that measures how simular predicted prompt is to real prompt by comparing n-gram overlap

- ROUGE-1: Counts overlap of individual words
- ROUGE-2: Counts overlap of word pairs
- ROUGE-L: Finds longest common sequence of words in order

In [32]:
model = T5ForConditionalGeneration.from_pretrained("prompt_predictor").to(torch.device("cuda"))
model.eval()

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
r1, r2, rL = [], [], []

for _, row in test_df.iterrows():
    # Tokenize input
    input_text = "predict prompt: " + row["answer"]
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(torch.device("cuda"))

    # Generate prediction
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )

    # Decode prediction back to text
    predicted = tokenizer.decode(output[0], skip_special_tokens=True)

    # Score predicted vs real prompt
    scores = scorer.score(row["prompt"], predicted)
    r1.append(scores["rouge1"].fmeasure)
    r2.append(scores["rouge2"].fmeasure)
    rL.append(scores["rougeL"].fmeasure)

# Print results
print(f"ROUGE-1:  {np.mean(r1):.4f}")
print(f"ROUGE-2:  {np.mean(r2):.4f}")
print(f"ROUGE-L:  {np.mean(rL):.4f}")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

ROUGE-1:  0.2577
ROUGE-2:  0.1496
ROUGE-L:  0.2248


### Sample Predictions

In [35]:
for _, row in test_df.head(5).iterrows():
    input_text = "predict prompt: " + row["answer"]
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(torch.device("cuda"))

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )

    predicted = tokenizer.decode(output[0], skip_special_tokens=True)

    print(f"\nAnswer (truncated): {row['answer'][:200]}")
    print(f"Real prompt:        {row['prompt']}")
    print(f"Predicted prompt:   {predicted}")
    print(f"ROUGE-L:            {scorer.score(row['prompt'], predicted)['rougeL'].fmeasure:.4f}")


Answer (truncated): I apologize if the previous response caused confusion. Here are the complete Basho and Wave classes, along with a full unit test file for the Basho class:Basho.js:[CODE_BLOCK_0]Wave.js:[CODE_BLOCK_1]b
Real prompt:        i hate this. write the files in full.
Predicted prompt:   Here are the complete Basho and Wave classes, along with the full unit test file for the Basho class.
ROUGE-L:            0.1481

Answer (truncated): [CODE_BLOCK_0]
Real prompt:        You are Junior, an AI system aiding developers.
You are working with a part of a large program called the "Working Set."
Before starting, check if you need more files to solve the task.
Do not edit files without knowing their contents!
Ask for them in normal conversational format instead.

# Working set

src/frontend/components/RequirementsEditor.jsx:
```
import { createSignal, createEffect } from 'solid-js';
import postDescriptor from '../service/postDescriptor';
import { promptDescriptor, setPromptDescriptor